# 2b. Dynamic Learning Trajectory, Uniform Distribution

**Scenario**: Synthetic animals with parameters that change across
sessions, mimicking a learning trajectory (random → active learning
→ expert). Uniform stimulus distribution throughout.

**Key question**: Can model ID on the expert-phase sessions still
identify the correct model despite the learning history? This
validates 3a's expert-only approach.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_NOTEBOOK_ROOT = _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _NOTEBOOK_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
if str(_NOTEBOOK_ROOT) not in sys.path:
	sys.path.insert(0, str(_NOTEBOOK_ROOT))
 
from shared_setup import *

import time
import pickle
from analysis.validation import (
    make_synthetic_cohort, make_learning_cohort, make_shift_cohort,
    run_gs_model_id, run_sbi_model_id, summarise_agreement,
)

SBI_OK = False
try:
    import torch
    SBI_OK = True
    print(f'SBI available (torch={torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

## 1. Configuration

In [ ]:
QUICK = True

if QUICK:
    N_SYNTHETIC = 3; N_GS_SEEDS = 2; BURN_IN = 500
    N_SBI_SIMS = 1_000; N_GENERIC_TRIALS = 300; N_CV_REPEATS = 4
else:
    N_SYNTHETIC = 5; N_GS_SEEDS = 4; BURN_IN = 1000
    N_SBI_SIMS = 20_000; N_GENERIC_TRIALS = 2500; N_CV_REPEATS = 32

SEED = 42
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

N_SESSIONS = 20        # total sessions (learning + expert)
TRIALS_PER_SESSION = 350
N_EXPERT = 8           # use last N sessions for model ID

## 2. Generate Learning-Trajectory Animals

In [ ]:
animals = make_learning_cohort(
    n_per_model=N_SYNTHETIC, n_sessions=N_SESSIONS,
    trials_per_session=TRIALS_PER_SESSION,
    burn_in=BURN_IN, seed=SEED,
)
print(f'{len(animals)} learning-trajectory animals')
for a in animals:
    accs = [s.stats(['accuracy'])['accuracy'] for s in a['sessions']]
    print(f'  {a["animal_id"]}: {a["true_model"]} '
          f'acc=[{accs[0]:.2f}→{accs[len(accs)//2]:.2f}→{accs[-1]:.2f}]')

# Add expert-only session key
for a in animals:
    a['expert_sessions'] = a['sessions'][-N_EXPERT:]

## 3. Model ID on Expert Sessions Only

Use only the last N sessions (expert phase) — this is exactly
what 3a does on real data.

In [ ]:
print('=== GS on expert sessions ===')
gs_df = run_gs_model_id(
    animals, sessions_key='expert_sessions',
    n_seeds=N_GS_SEEDS, burn_in=BURN_IN,
)

In [ ]:
sbi_df = pd.DataFrame()
if SBI_OK:
    print('=== SBI on expert sessions ===')
    sbi_df = run_sbi_model_id(
        animals, sessions_key='expert_sessions',
        n_sbi_sims=N_SBI_SIMS, n_generic_trials=N_GENERIC_TRIALS,
        n_cv_repeats=N_CV_REPEATS, burn_in=BURN_IN, seed=SEED,
    )

## 4. Agreement

In [ ]:
merged = summarise_agreement(gs_df, sbi_df, 'Dynamic Learning, Expert-Only ID')
print()
print(merged.to_string(index=False))

## 5. Save

In [ ]:
# Save results for 2e agreement summary
save_path = Path('../results/validation/2b_dynamic_uniform.pkl')
save_path.parent.mkdir(parents=True, exist_ok=True)
with open(save_path, 'wb') as f:
    pickle.dump({'gs': gs_df, 'sbi': sbi_df if SBI_OK else None,
                'scenario': '2b_dynamic_uniform'
}, f)
print(f'Saved to {save_path}')